# NOVA Strong Model — Qwen3-8B QLoRA

This optional experiment **actually fine-tunes** an Apache-2.0 8.2-billion-parameter pretrained text model when run on an available NVIDIA GPU. It is a major upgrade from the original 137K-parameter NOVA Core training architecture, but the dataset is **only 77 tiny demonstrations**. It won't match ChatGPT or Claude, and the resulting adapter will **not** appear in your live NOVA web app without a compatible inference host.

### Before running
Choose **Runtime → Change runtime type → GPU**. A 16 GB T4 is a possible minimum target for low-memory 4-bit LoRA at 512 tokens, but availability and actual successful fit are not guaranteed. Google Colab sets free quotas. No paid model or provider is automatically activated.

Model card: https://huggingface.co/Qwen/Qwen3-8B


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime in Colab first; this notebook cannot train 8B on CPU.'
print('Device:', torch.cuda.get_device_name(0))
print('Free VRAM GiB:', round(torch.cuda.mem_get_info()[0]/2**30,1))
!pip -q install -U 'trl>=0.21' 'peft>=0.17' 'transformers>=4.55' 'datasets>=3.6' 'accelerate>=1.5' 'bitsandbytes>=0.45' 'safetensors>=0.4'


In [ ]:
import os
if not os.path.exists('/content/NOVA-AI/.git'):
    !git clone --depth 1 https://github.com/Exclusive2knight/NOVA-AI.git /content/NOVA-AI
%cd /content/NOVA-AI
!python training/build_nova_sft.py --input training/nova_sft_examples.jsonl --out-dir nova-data/sft


In [ ]:
# Start with a conservative 30-step QLoRA test; it does not imply frontier performance.
!python training/finetune_qwen3_8b_qlora.py --data-dir nova-data/sft --out-dir nova-data/nova-qwen3-8b-qlora --max-steps 30


## Export adapter and training report

This downloads only a small LoRA adapter, **not** the full pretrained Qwen3-8B weights. Store privately and never commit private training data. Compare baseline and tuned answers separately before deploying to a suitable self-hosted inference service.


In [ ]:
import shutil
from google.colab import files
archive=shutil.make_archive('/content/NOVA_Strong_8B_LoRA','zip','nova-data/nova-qwen3-8b-qlora')
print('Adapter saved:',archive)
files.download(archive)
